<a href="https://colab.research.google.com/github/raj-027/Sanskrit-NLP/blob/main/whisper_transcription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torchaudio librosa

In [ ]:
from transformers import pipeline

# Load ASR pipeline
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    device=0,  # use -1 for CPU
    chunk_length_s=30 # Explicitly enable chunking
)

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [ ]:
import librosa

audio_path = "/content/drive/MyDrive/Sanskrit NLP/audio_data/RigVeda-1-1-4-VedaBharati.mp3"

# Load audio
audio, sr = librosa.load(audio_path, sr=16000)

In [ ]:
result = pipe({"sampling_rate": sr, "raw": audio}, generate_kwargs={"language": "hi"})
print("Transcription:")
print(result["text"])


Loading Rigveda text

In [ ]:
def process_sanskrit_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        # Read the entire file content
        content = file.read()

        # Remove all hyphens
        cleaned_text = content.replace('-', '')
        cleaned_text = content.replace('_', '')

        # Convert to a list of lines, removing any extra whitespace
        sanskrit_list = [line.strip() for line in cleaned_text.splitlines() if line.strip()]

    return sanskrit_list

# Usage
my_list = process_sanskrit_file('/content/drive/MyDrive/Sanskrit NLP/161655_RV_Terms.txt')
print(my_list)


In [ ]:
import json

def create_and_print_sanskrit_map(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            # Clean underscores and hyphens while maintaining original as key
            sanskrit_dict = {
                line.strip(): line.replace('-', '').replace('_', '').strip()
                for line in file if line.strip()
            }

        # Print the entire dictionary in a formatted way
        # ensure_ascii=False is critical to display Devanagari correctly
        print(json.dumps(sanskrit_dict, indent=4, ensure_ascii=False))

        return sanskrit_dict
    except FileNotFoundError:
        print("Error: The file 'sanskrit_texts.txt' was not found.")

# Usage
sanskrit_map = create_and_print_sanskrit_map('/content/drive/MyDrive/Sanskrit NLP/161655_RV_Terms.txt')
print(len(sanskrit_map))


{
    "असा॑वि": "असा॑वि",
    "सोमः": "सोमः",
    "इन्द्र": "इन्द्र",
    "ते॒": "ते॒",
    "शवि॑ष्ठ": "शवि॑ष्ठ",
    "धृष्णो इति": "धृष्णो इति",
    "आ": "आ",
    "गहि": "गहि",
    "त्वा": "त्वा",
    "पृणक्तु": "पृणक्तु",
    "इन्द्रियम्": "इन्द्रियम्",
    "रजः॑": "रजः॑",
    "सूर्यः॑": "सूर्यः॑",
    "न": "न",
    "र": "र",
    "इन्द्रम्": "इन्द्रम्",
    "इत्": "इत्",
    "हरी॒ इति॑": "हरी॒ इति॑",
    "व॒ह॒तः॒": "व॒ह॒तः॒",
    "अप्र॑तिधृष्टऽशवसम्": "अप्र॑तिधृष्टऽशवसम्",
    "ऋषीणाम्": "ऋषीणाम्",
    "च": "च",
    "स्तुतीः": "स्तुतीः",
    "उप॑": "उप॑",
    "यज्ञम्": "यज्ञम्",
    "मानु॑षाणाम्": "मानु॑षाणाम्",
    "तिष्ठ": "तिष्ठ",
    "वृत्र-हन्": "वृत्रहन्",
    "रथम्": "रथम्",
    "यु॒क्ता": "यु॒क्ता",
    "ते": "ते",
    "ब्रह्मणा": "ब्रह्मणा",
    "अर्वाचीनम्": "अर्वाचीनम्",
    "सु": "सु",
    "मनः": "मनः",
    "ग्रावा": "ग्रावा",
    "कृणोतु": "कृणोतु",
    "व॒ग्नुना॑": "व॒ग्नुना॑",
    "इमम्": "इमम्",
    "सुतम्": "सुतम्",
    "पिब": "पिब",
    "ज्येष्ठम्": "ज्येष्ठम्",
   

In [ ]:
import unicodedata

DEVANAGARI_MATRAS = {
    "ा": "आ", "ि": "इ", "ी": "ई", "ु": "उ", "ू": "ऊ",
    "ृ": "ऋ", "ॄ": "ॠ", "ॢ": "ऌ", "ॣ": "ॡ",
    "े": "ए", "ै": "ऐ", "ो": "ओ", "ौ": "औ",
}

# independent vowels set
DEVANAGARI_VOWELS = set(list("अआइईउऊऋॠऌॡएऐओऔ"))

# consonants range roughly (क..ह) — we'll treat these as consonants
# include nukta & other combining marks handled separately
DEVANAGARI_CONSONANTS = set(list(
    "कखगघङचछजझञटठडढणतथदधनपफबभमयऱलवशषसह"
))
# add retroflex/rule variants if needed (adjust per data)
# Halant, nukta, anusvara, visarga
HALANT = "\u094D"     # ्
NUKTA = "\u093C"      # ़
ANUSVARA = "ं"
VISARGA = "ः"
CANDRABINDU = "ँ"

PHONETIC_MODIFIERS = {ANUSVARA, VISARGA, CANDRABINDU}

# Add non-phonemic characters to ignore
NON_PHONEMIC_CHARS_TO_IGNORE = {
    " ",          # space
    "\u0951",     # DEVANAGARI ACCENT ACUTE
    "\u0952",     # DEVANAGARI ACCENT GRAVE
    "\u093d",     # DEVANAGARI ABBREVIATION SIGN
    "\u0950",     # DEVANAGARI OM
    "\u0900",     # DEVANAGARI SIGN ANUSVARA (variant of anusvara, might appear)
    "\u0964",     # DEVANAGARI DANDA
    "\u0965",     # DEVANAGARI DOUBLE DANDA
    "ऽ",         # Avagraha
    "|"          # Pipe symbol, sometimes used in texts
}

def akshara_to_phonemes(token):

    token = unicodedata.normalize("NFC", token.strip())
    phonemes = []
    i = 0
    chars = list(token)

    while i < len(chars):
        ch = chars[i]

        # Ignore known non-phonemic characters
        if ch in NON_PHONEMIC_CHARS_TO_IGNORE:
            i += 1
            continue

        # independent vowel
        if ch in DEVANAGARI_VOWELS:
            phonemes.append(ch)
            i += 1
            continue

        # modifier symbols that act like separate phonemes (anusvara/visarga)
        if ch in PHONETIC_MODIFIERS:
            phonemes.append(ch)
            i += 1
            continue

        # consonant (including possible nukta immediately after)
        if ch in DEVANAGARI_CONSONANTS:
            base = ch
            i += 1
            # nukta (rare) e.g. क़
            if i < len(chars) and chars[i] == NUKTA:
                base = base + chars[i]
                i += 1

            # halant means explicit consonant without inherent vowel
            if i < len(chars) and chars[i] == HALANT:
                # append base (consonant) only, skip halant
                phonemes.append(base)
                i += 1
                continue

            # vowel matra attached? map to independent vowel and append base+vowel
            if i < len(chars) and chars[i] in DEVANAGARI_MATRAS:
                mat = chars[i]
                vowel = DEVANAGARI_MATRAS[mat]
                phonemes.append(base)
                phonemes.append(vowel)
                i += 1
                continue

            # If no matra/halant follows, append consonant (in many phonemic analyses
            # the inherent vowel 'अ' is present. Depending on your phoneme vectors you
            # may want to append 'अ' as well. Here we append the consonant alone,
            # which matches a common phoneme mapping where consonant segments are separate.)
            phonemes.append(base)
            continue

        # standalone matra (shouldn't usually happen), convert to vowel
        if ch in DEVANAGARI_MATRAS:
            phonemes.append(DEVANAGARI_MATRAS[ch])
            i += 1
            continue

        # otherwise: unknown char, append as-is (fallback)
        phonemes.append(ch)
        i += 1

    return phonemes

loading the model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class TripletModelEncoder(nn.Module):
    def __init__(self, input_dim,
                 hidden_dim=128,
                 layers=2,
                 proj_dim=128):
        super().__init__()

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=layers,
            batch_first=True,
            bidirectional=True
        )

        self.attn = nn.Linear(hidden_dim * 2, 1)

        self.proj = nn.Linear(hidden_dim * 2, proj_dim)

    def forward(self, x, lengths):

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)

        # mask for padded positions
        mask = torch.arange(out.size(1)).to(lengths.device)[None, :] < lengths[:, None]

        scores = self.attn(out).squeeze(-1)   # [batch, seq_len]
        scores = scores.masked_fill(~mask, -1e9)

        weights = torch.softmax(scores, dim=1)  # [batch, seq_len]
        pooled = torch.sum(out * weights.unsqueeze(-1), dim=1)

        return self.proj(pooled)

In [ ]:
INPUT_DIM = 34
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


In [ ]:
triplet_model = TripletModelEncoder(
    input_dim=INPUT_DIM,
    hidden_dim=256,
    layers=2,
    proj_dim=128
).to(device)

triplet_model.load_state_dict(
    torch.load("/content/drive/MyDrive/Sanskrit NLP/models/TML_model.pkl", map_location=device)
)

triplet_model.eval()
print("Triplet model loaded.")

Triplet model loaded.


get embedding

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Sanskrit NLP/sanskrit_phoneme_vectors (1).csv", index_col=0)

phoneme_vectors = {
    row.Index: np.array(row[1:], dtype=float)
    for row in df.itertuples()
}

print(f"Loaded {len(phoneme_vectors)} Sanskrit phonemes with 34 features.")


Loaded 52 Sanskrit phonemes with 34 features.


In [ ]:
def word_to_feature_tensor(word):

    phonemes = akshara_to_phonemes(word)

    vectors = []

    for ph in phonemes:

        if ph in phoneme_vectors:
            vec = phoneme_vectors[ph]
        else:
            print("Unknown phoneme:", ph)
            vec = np.zeros(34)

        vectors.append(vec)

    tensor = torch.tensor(np.array(vectors), dtype=torch.float32)

    return tensor

In [ ]:
import torch.nn.functional as F

def get_embeddings(model, words, is_contrastive_model=False, normalize=True):

    embeddings = {}
    model.eval()

    with torch.no_grad():

        for w in words:

            feature_tensor = word_to_feature_tensor(w).to(device)
            feature_tensor = feature_tensor.unsqueeze(0)

            length = torch.tensor(
                [feature_tensor.size(1)],
                dtype=torch.long,
                device=feature_tensor.device
            )

            if is_contrastive_model:
                vec = model(feature_tensor)
            else:
                vec = model(feature_tensor, length)

            vec = vec.squeeze(0)

            if normalize:
                vec = F.normalize(vec, p=2, dim=0)

            embeddings[w] = vec.cpu().numpy()

    return embeddings

normalize the embeddings

In [ ]:
def normalize(vec):
    norm = np.linalg.norm(vec)
    return vec / norm if norm != 0 else vec

In [ ]:
corpus_embeddings = {}

# Get embeddings for all processed words once
all_processed_words = list(sanskrit_map.values())
# get_embeddings function already normalizes the embeddings if normalize=True
all_embeddings_dict = get_embeddings(triplet_model, all_processed_words, normalize=True)

# Populate corpus_embeddings, mapping original words to their corresponding embeddings
for original_word, processed_word in sanskrit_map.items():
    if processed_word in all_embeddings_dict:
        corpus_embeddings[original_word] = all_embeddings_dict[processed_word]
    else:
        # This handles cases where a processed_word might not have an embedding,
        # though ideally all should be present if `all_processed_words` was comprehensive.
        print(f"Warning: Processed word '{processed_word}' (from original '{original_word}') not found in computed embeddings.")

In [ ]:
def preprocess(word):
    word = word.replace('-', '').replace('_', '').strip()

    # remove halant
    word = word.replace("्", "")

    return word

In [ ]:
def cosine_sim(a, b):
    return np.dot(a, b)

In [ ]:
def find_closest_word(query_vec, corpus_embeddings):
    best_word = None
    best_score = -1

    for word, vec in corpus_embeddings.items():
        score = cosine_sim(query_vec, vec)

        if score > best_score:
            best_score = score
            best_word = word

    return best_word, best_score

In [ ]:
def correct_word(word, corpus_embeddings, threshold=0.85):

    processed_word = preprocess(word)

    # Get embedding for the single word, it will be normalized by get_embeddings
    query_embedding_dict = get_embeddings(triplet_model, [word], normalize=True)
    query_vec = query_embedding_dict[word] # Extract the embedding for the word

    best_word, score = find_closest_word(query_vec, corpus_embeddings)

    print(f"{word} → {best_word} (score={score:.3f})")  # debug

    if score >= threshold:
        return best_word
    else:
        return word

In [ ]:
def correct_sentence(text, corpus_embeddings, threshold=0.85):

    words = text.split()

    corrected_words = [
        correct_word(w, corpus_embeddings, threshold)
        for w in words
    ]

    return " ".join(corrected_words)

In [ ]:
input_text = "अग्निफ्पूर्वे भिर रिशि"

output = correct_sentence(input_text, corpus_embeddings)

print("Corrected:", output)